In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import mlflow
import mlflow.sklearn

In [0]:
df = spark.read.table("workspace.default.capstone_trade_model_data").toPandas()

In [0]:
#df["profitable"] = (df["pnl_price"] > 0).astype(int)
df = df.dropna()

In [0]:
drop_columns = ["strategy", "entry_date", "exit_date", "pnl_price", "profitable", "Close", "Low", "High", "Date"]

X_columns = [col for col in df.columns if col not in drop_columns]

X = df[X_columns]
y = df["profitable"]

In [0]:
strategies = df["strategy"].dropna().unique()

In [0]:
summary_results = []
all_cv_results = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for strat in strategies:
    print(f"\nRunning strategy: {strat}")
    
    df_strat = df[df["strategy"] == strat].copy()

    # Target
    y_strat = df_strat["profitable"]

    # Start with chosen columns
    X_strat = df_strat[X_columns].copy()

    # Keep only numeric columns
    X_strat = X_strat.select_dtypes(include=["number"])

    # Replace inf values with NaN so the imputer can handle them
    X_strat = X_strat.replace([np.inf, -np.inf], np.nan)

    print("Features used:")
    print(X_strat.columns.tolist())

    # Safety checks
    if len(df_strat) < 10:
        print(f"Skipping {strat}: too few rows")
        continue

    if y_strat.nunique() < 2:
        print(f"Skipping {strat}: only one class in target")
        continue

    if y_strat.value_counts().min() < 2:
        print(f"Skipping {strat}: one class has fewer than 2 rows")
        continue

    if X_strat.shape[1] == 0:
        print(f"Skipping {strat}: no numeric features available")
        continue

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_strat,
        y_strat,
        test_size=0.20,
        random_state=42,
        stratify=y_strat
    )

    # Make sure 5-fold CV is possible on the training set
    train_class_counts = y_train.value_counts()
    if train_class_counts.min() < 5:
        print(f"Skipping {strat}: not enough samples per class for 5-fold CV")
        continue

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ])

    param_grid = {
        "model__C": [0.01, 0.1, 1, 10],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"]
    }

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=cv,
        scoring="f1",
        n_jobs=-1,
        error_score="raise",
        return_train_score=True
    )

    with mlflow.start_run(run_name=f"logreg_{strat}"):
        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test)
        y_prob = best_model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)

        print("Best params:", grid.best_params_)
        print("Best CV F1:", round(grid.best_score_, 4))
        print("Test Accuracy:", round(acc, 4))
        print("Test F1:", round(f1, 4))
        print("Test AUC:", round(auc, 4))
        print(classification_report(y_test, y_pred))

        cv_results_df = pd.DataFrame(grid.cv_results_).copy()
        cv_results_df["strategy"] = strat

        print(f"\nGrid search results for {strat}:")
        print(
            cv_results_df[
                ["rank_test_score", "mean_test_score", "std_test_score", "mean_train_score", "params"]
            ].sort_values("rank_test_score")
        )

        all_cv_results.append(cv_results_df)

        summary_results.append({
            "strategy": strat,
            "best_params": str(grid.best_params_),
            "best_cv_f1": grid.best_score_,
            "test_accuracy": acc,
            "test_f1": f1,
            "test_auc": auc
        })

        mlflow.log_params(grid.best_params_)
        mlflow.log_param("strategy", strat)
        mlflow.log_param("test_size", 0.20)

        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_f1", f1)
        mlflow.log_metric("test_auc", auc)
        mlflow.log_metric("best_cv_score", grid.best_score_)

        mlflow.sklearn.log_model(best_model, artifact_path=f"model_{strat}")

summary_df = pd.DataFrame(summary_results)
all_cv_results_df = pd.concat(all_cv_results, ignore_index=True)

print("\nOverall strategy summary:")
print(summary_df.sort_values("test_f1", ascending=False))

summary_df.to_csv("logreg_strategy_summary.csv", index=False)

all_cv_results_df["params"] = all_cv_results_df["params"].astype(str)
all_cv_results_df.to_csv("logreg_all_gridsearch_results.csv", index=False)

spark.createDataFrame(summary_df).write.mode("overwrite").format("delta").saveAsTable(
    "workspace.default.logreg_strategy_summary"
)

spark.createDataFrame(all_cv_results_df).write.mode("overwrite").format("delta").saveAsTable(
    "workspace.default.logreg_all_gridsearch_results"
)

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

experiment = mlflow.get_experiment_by_name("/Users/schylar.srey@uhsp.edu/StockMarket_CapstoneProject/CapstoneProject/Milestone 4/Machine Learning Models/Logistic Regression 2026-03-22 16_13_00")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

for _, row in runs.iterrows():
    run_id = row["run_id"]
    strat = row["params.strategy"]

    print(f"\nStrategy: {strat}")

    model_uri = f"runs:/{run_id}/model_{strat}"
    model = mlflow.sklearn.load_model(model_uri)

    # Recreate test data 
    df_strat = df[df["strategy"] == strat]

    X = df_strat[X_columns].select_dtypes(include=["number"]).replace([np.inf, -np.inf], np.nan)
    y = df_strat["profitable"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:\n", cm)